# Assignment 4
## Econ 8310 - Business Forecasting

This assignment will make use of the bayesian statistical models covered in Lessons 10 to 12. 

A/B Testing is a critical concept in data science, and for many companies one of the most relevant applications of data-driven decision-making. In order to improve product offerings, marketing campaigns, user interfaces, and many other user-facing interactions, scientists and engineers create experiments to determine the efficacy of proposed changes. Users are then randomly assigned to either the treatment or control group, and their behavior is recorded.
If the changes that the treatment group is exposed to can be measured to have a benefit in the metric of interest, then those changes are scaled up and rolled out to across all interactions.
Below is a short video detailing the A/B Testing process, in case you want to learn a bit more:
[https://youtu.be/DUNk4GPZ9bw](https://youtu.be/DUNk4GPZ9bw)

For this assignment, you will use an A/B test data set, which was pulled from the Kaggle website (https://www.kaggle.com/datasets/yufengsui/mobile-games-ab-testing). I have added the data from the page into Codio for you. It can be found in the cookie_cats.csv file in the file tree. It can also be found at [https://github.com/dustywhite7/Econ8310/raw/master/AssignmentData/cookie_cats.csv](https://github.com/dustywhite7/Econ8310/raw/master/AssignmentData/cookie_cats.csv)

The variables are defined as follows:

| Variable Name  | Definition |
|----------------|----|
| userid         | A unique number that identifies each player  |
| version        | Whether the player was put in the control group (gate_30 - a gate at level 30) or the group with the moved gate (gate_40 - a gate at level 40) |
| sum_gamerounds | The number of game rounds played by the player during the first 14 days after install.  |
| retention1     | Did the player come back and play 1 day after installing?     |
| retention7     | Did the player come back and play 7 days after installing?    |               

### The questions

You will be asked to answer the following questions in a small quiz on Canvas:
1. What was the effect of moving the gate from level 30 to level 40 on 1-day retention rates?
2. What was the effect of moving the gate from level 30 to level 40 on 7-day retention rates?
3. What was the biggest challenge for you in completing this assignment?

You will also be asked to submit a URL to your forked GitHub repository containing your code used to answer these questions.

In [ ]:
# Import all at once
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az

In [ ]:
# Data imports and exploring
df = pd.read_csv("https://github.com/dustywhite7/Econ8310/raw/master/AssignmentData/cookie_cats.csv")
df['treatment'] = (df['version'] == 'gate_40').astype(int)
df['gamerounds_std'] = (df['sum_gamerounds'] - df['sum_gamerounds'].mean()) / df['sum_gamerounds'].std() # gamerounds will be used as a control variable; need normalization before being used
df.head(6)

In [ ]:
# Subsampling
# Original data consist of 90,189 raws; Too much running time
df = df.sample(10000, random_state=42).reset_index(drop=True)

In [ ]:
# Modeling Retention_1
coords = {"observation": df.index.values}
with pm.Model(coords=coords) as bayesian_model:
    treatment  = pm.Data("treatment",  df['treatment'].values,  dims="observation")
    gamerounds = pm.Data("gamerounds", df['gamerounds_std'].values,   dims="observation")
    retention1 = pm.Data("retention1", df['retention_1'].values, dims="observation")
    # Priors (normal distribution)
    β0         = pm.Normal("β0",         mu=0, sigma=1)
    β_treatment = pm.Normal("β_treatment", mu=0, sigma=1)
    β_gamerounds  = pm.Normal("β_gamerounds",  mu=0, sigma=1)
    # Linear model
    μ = β0 + β_treatment * treatment + β_gamerounds * gamerounds #gamerounds as a control variable
    p = pm.math.sigmoid(μ)
    likelihood = pm.Bernoulli("y", p=p, observed=retention1, dims="observation")

# Fit & sample
with bayesian_model:
    trace = pm.sample(5000, return_inferencedata=True, target_accept=0.9) #NUTS instead of Metropolis
burned_trace = trace.sel(draw=slice(500, None))

# Results & Plots
print(az.summary(burned_trace, var_names=["β0", "β_treatment", "β_gamerounds"]))
az.plot_posterior(burned_trace, var_names=["β0", "β_treatment", "β_gamerounds"])

In [ ]:
# Modeling Retention_7
coords = {"observation": df.index.values}
with pm.Model(coords=coords) as bayesian_model:
    treatment  = pm.Data("treatment",  df['treatment'].values,  dims="observation")
    gamerounds = pm.Data("gamerounds", df['gamerounds_std'].values,   dims="observation")
    retention7 = pm.Data("retention7", df['retention_7'].values, dims="observation")
    # Priors (normal distribution)
    β0         = pm.Normal("β0",         mu=0, sigma=1)
    β_treatment = pm.Normal("β_treatment", mu=0, sigma=1)
    β_gamerounds  = pm.Normal("β_gamerounds",  mu=0, sigma=1)
    # Linear model
    μ = β0 + β_treatment * treatment + β_gamerounds * gamerounds #gamerounds as a control variable
    p = pm.math.sigmoid(μ)
    likelihood = pm.Bernoulli("y", p=p, observed=retention7, dims="observation")

# Fit & sample
with bayesian_model:
    trace = pm.sample(5000, return_inferencedata=True, target_accept=0.9) #NUTS instead of Metropolis
burned_trace = trace.sel(draw=slice(500, None))

# Results & Plots
print(az.summary(burned_trace, var_names=["β0", "β_treatment", "β_gamerounds"]))
az.plot_posterior(burned_trace, var_names=["β0", "β_treatment", "β_gamerounds"])